# Getting Started with Automated-LLM-Probes

In [1]:
from __future__ import annotations
import os
from pathlib import Path
import automated_intelligence_tests as ait
import automated_llm_probes as alp
assert os.environ.get("OPENAI_API_KEY")

### Models available

In [2]:
models = alp.ready_models()
print(f"|Models available: {len(models)}|\n")
for m in models:
    print(f"  {m['name']:20s} {m['api']:12s} {m['model_id']}")

|Models available: 56|

  GPT-3.5-Turbo        openai       gpt-3.5-turbo
  Moonshot-v1-8k       moonshot     moonshot-v1-8k
  Moonshot-v1-128k     moonshot     moonshot-v1-128k
  GPT-4o               openai       gpt-4o-2024-08-06
  Qwen-Turbo           qwen         qwen-turbo
  o4-mini              openai       o4-mini-2025-04-16
  GPT-4.1              openai       gpt-4.1-2025-04-14
  GPT-4.1-mini         openai       gpt-4.1-mini-2025-04-14
  GPT-4.1-nano         openai       gpt-4.1-nano-2025-04-14
  Kimi-K2              moonshot     moonshot-v1-32k
  Qwen3-235B-Instruct  qwen         qwen3-235b-a22b-instruct-2507
  GPT-5                openai       gpt-5-2025-08-07
  GPT-5-mini           openai       gpt-5-mini-2025-08-07
  Qwen-Max             qwen         qwen-max
  Qwen-Plus            qwen         qwen-plus
  Claude Sonnet 4.5    claude       claude-sonnet-4-5-20250929
  MiniMax-M2.5         openrouter   minimax/minimax-m2.5
  Claude Haiku 4.5     claude       claude-haiku-4-

### Probed tasks

In [3]:
path = Path('./data/'); n=1
print(f"|Probed tasks: {len([p for p in path.iterdir() if p.is_dir()])}|\n")
for d in path.iterdir():
    if d.is_dir():
        count = sum(1 for f in d.rglob('*') if f.is_file())
        print(f"  {n}. {d.name.upper()[:7]:8}:  {count}"); n+=1

|Probed tasks: 3|

  1. AUT     :  6067
  2. DAT     :  6286
  3. WRT     :  5335


### Tests availabe:

In [4]:
ait_counts = ait.list_available_tests()
print(f'|Available tests: {len(ait_counts)}|\n')
for i,v in ait_counts.items():
    print(f'  {i} : {v}')

|Available tests: 4|

  AUT : Alternative Uses Task
  CAT : Convergent Association Task
  DAT : Divergent Association Task
  WRT : Creative Writing Task


## Tiny trial collection (DAT, 2 responses)

In [5]:
def test_model(test_name="DAT", model_name="GPT-3.5-Turbo", n=250):
    models = [m for m in alp.ready_models() if m["name"] == model_name]
    alp.collect(test_name, models=models, n_per_model=n)

test_model("DAT", n=265)

  GPT-3.5-Turbo: 300/265 done — skip


## Parse & merge

In [6]:
from __future__ import annotations
import re, pandas as pd
import glove_word_embeddings as gwe

def parse_dat(raw):
    text = str(raw or "").strip().strip('"').strip("'")
    tokens = re.split(r"[,\n\r]+", text)
    nouns = [n for n in (gwe.pre.clean_word(t) for t in tokens) if n][:10]
    return nouns + [""] * (10 - len(nouns))

def parse_aut(raw):
    uses = []
    for line in re.split(r"[\n\r]+", str(raw or "")):
        line = re.sub(r"^\s*[\d\.\)\-]+\s*", "", line)
        if toks := [t for t in (gwe.pre.clean_word(t) for t in line.split()) if t]:
            uses.append(" ".join(toks))
    return ", ".join(uses)

def parse_wrt(raw):
    text = re.sub(r"^#+\s*.*$", "", str(raw or ""), flags=re.M)
    text = re.sub(r"^\s*Title:.*$", "", text, flags=re.M | re.I)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def load_task(task: str) -> pd.DataFrame:
    task = task.lower()
    df = pd.DataFrame.from_dict(alp.parse_and_merge(task),orient='index')
    if task == "dat":
        parsed = df["raw"].map(parse_dat)
        df[[f"noun_{i}" for i in range(10)]] = parsed.tolist()
        df["response_clean"] = parsed.map(lambda x: ", ".join(n for n in x if n))
        extra = [f"noun_{i}" for i in range(10)]
    elif task == "aut":
        df["object"] = df["prompt"].str.extract(
            r"object: (.+?)\?", expand=False).str.strip()
        df["response_clean"] = df["raw"].map(parse_aut)
        extra = ["object"]
    elif task == "wrt":
        cues = df["prompt"].str.extract(
            r"words: (.+?)\.", expand=False).str.strip().str.split(r",\s*")
        df[["cue_0", "cue_1", "cue_2"]] = pd.DataFrame(cues.tolist()).iloc[:, :3]
        df["response_clean"] = df["raw"].map(parse_wrt)
        extra = ["cue_0", "cue_1", "cue_2"]
    else:
        raise ValueError(f"Unknown task: {task}")
    cols = ["task", "model_name", "model_id", 
            "provider", "rep", "temperature_std"] + extra + [
        "prompt", "response_clean", "ts_utc", "hash"]
    return df[[c for c in cols if c in df.columns]].sort_values(
        ["model_name", "rep"]).reset_index(drop=True)

def load_tasks():
    for task in ("dat", "aut", "wrt"):
        print(f"Parsing {task.upper()}...")
        df = load_task(task)
        print(df.shape)
        df.to_csv(f"./data/{task}.csv", index=False)

load_task('aut')

aut: 100%|██████████████████████████████████████████████████████████| 6069/6069 [00:39<00:00, 155.06it/s]


,task,model_name,model_id,provider,rep,temperature_std,object,prompt,response_clean,ts_utc
0,AUT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,0,0.5,brick,What are some creative uses for this object: b...,"doorstop personality any room, weight holding ...",2026-08-15T13:46:49.221432+00:00
1,AUT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,1,0.5,paperclip,What are some creative uses for this object: p...,"emergency fish hook survival situations, bookm...",2026-08-15T13:46:56.831422+00:00
2,AUT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,2,0.5,bucket,What are some creative uses for this object: b...,"diy drum kit making music, planter growing veg...",2026-08-15T13:47:00.885017+00:00
3,AUT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,3,0.5,sock,What are some creative uses for this object: s...,"puppet storytelling entertaining children, dus...",2026-08-15T13:47:04.950421+00:00
4,AUT,Claude Haiku 4.5,claude-haiku-4-5-20251001,anthropic,4,0.5,fork,What are some creative uses for this object: f...,"plant stake small potted plants, hair pick tex...",2026-08-15T13:47:09.163931+00:00
...,...,...,...,...,...,...,...,...,...,...
6042,AUT,Llama-4 Scout,meta-llama/llama-4-scout,openrouter,245,0.5,purse,What are some creative uses for this object: p...,"plant holder, gift wrapping, diy puppet theate...",2026-08-16T03:15:00.727854+00:00
6043,AUT,Llama-4 Scout,meta-llama/llama-4-scout,openrouter,246,0.5,comb,What are some creative uses for this object: c...,"plant label, diy craft tool, hairpiece, pencil...",2026-08-16T03:15:01.757694+00:00
6044,AUT,Llama-4 Scout,meta-llama/llama-4-scout,openrouter,247,0.5,baseball,What are some creative uses for this object: b...,"paperweight, bookend, dog toy, decorative item...",2026-08-16T03:15:02.907469+00:00
6045,AUT,Llama-4 Scout,meta-llama/llama-4-scout,openrouter,248,0.5,candle,What are some creative uses for this object: c...,"diy home fragrance, emergency light source, am...",2026-08-16T03:15:03.961790+00:00
